In [ ]:
using WAVI 
using Printf
using ImageFiltering
using SparseArrays


In [ ]:
#
#Grid and boundary conditions
#
nx = 664 # use the clipped grid
ny = 553 # use the clipped grid
nσ = 26
x0 = -2764000.0
y0 = -2260000.0
dx = 8000.0
dy = 8000.0

path = "/home/tgia02/repos/WAVIConstructor.jl/examples/outputs/full_antarctica/"

In [ ]:
sigma_grid=Array{Float64}(undef,nσ);
read!(path*"sigma_grid.bin",sigma_grid)

In [ ]:
sigma_grid

In [ ]:
#This is a test of the inversion process with MISMIP_PLUS
using WAVI 
using ImageFiltering
using SparseArrays
using IterativeSolvers

function MISMIP_PLUS_inversion(simulation)

   model = simulation.model

    # Read in data to be used for the inversion. This is taken from the MISMIP_PLUS output. 
    # The velocity data is calculated on the h-grid and returned on the u,v grid so doesn't need to be smoothed for use in the inversion.

    #pre-select points not near data gaps e.g. mask edge. Domain goes to ice shelf front so remove last row  
    #mask out first and last rows.
    speed_u_mask=deepcopy(simulation.model.fields.gu.mask)
    speed_u_mask[1:2, :] .= false
    speed_u_mask[end, :] .= false
    speed_v_mask=deepcopy(simulation.model.fields.gv.mask)
    speed_v_mask[:, 2] .= false
    speed_v_mask[:, end-1] .= false

    data = DataFields(grid = deepcopy(simulation.model.grid),
                              speed_u = deepcopy(simulation.model.fields.gu.us),
                              speed_u_mask =  speed_u_mask,
                              speed_v = deepcopy(simulation.model.fields.gv.vs),
                              speed_v_mask =  speed_v_mask,
                              dhdt = deepcopy(simulation.model.fields.gh.dhdt),
                              accumulation_rate = deepcopy(simulation.model.params.accumulation_rate),
                              dhdtacc_mask=deepcopy(simulation.model.fields.gh.mask))

    reltol=0.5
    abstol=0.1
    maxiter=500
    gmres_restart =50
    βgrounded_start=1.0e3
    βfloating_start=1.0e-10
    ηstart_guess = 1.0e6
    βpower = 0.1
    Bpower_shelf = 0.1
    Bpower_grounded = 0.0
    cg=false
    gmres=true

    inversion_params = InversionParams(reltol = reltol,
                                    maxiter = maxiter,
                                    abstol = abstol,
                                    gmres_restart = gmres_restart,
                                    βgrounded_start = βgrounded_start,
                                    βfloating_start = βfloating_start,
                                    ηstart_guess = ηstart_guess,
                                    βpower = βpower,
                                    Bpower_grounded = Bpower_grounded,
                                    Bpower_shelf = Bpower_shelf,
                                    cg = cg,
                                    gmres = gmres)
                                
    #JKVstepping parameters
    niter0 = 0
    #n_iter_out=500
    max_JKV_iterations = 300
    #n_iter_chkpt = 1000
    #n_iter_pchkpt= 100

    JKVstepping_params = JKVsteppingParams(niter0 = niter0, 
                                   #     n_iter_chkpt = n_iter_chkpt,
                                    #    n_iter_pchkpt = n_iter_pchkpt,
                                        n_iter_total = max_JKV_iterations)
                                      #  n_iter_out = n_iter_out)

    inversion_output = InversionOutput()
   
    dirichlet_model = deepcopy(simulation.model)

    inversion = Inversion(model = dirichlet_model,
                        data = data,
                        inversion_params=inversion_params,
                        inversion_output = inversion_output)
                            
    inversion_simulation = InversionSimulation(model = model, 
                                        inversion = inversion,
                                        JKVstepping_params = JKVstepping_params)
           
    run_inversion_simulation!(inversion_simulation)

    return inversion_simulation

end
